# 20. 테스트 노드 중복 적재 실패 경로 확인

- 같은 chunk_id를 두 번 넣었을 때 두 번째가 실패하는지 본다 (CREATE와 유일 제약).
- 실패한 적재가 기존 테스트 노드를 지우지 않는지 확인한다. 정리는 첫 번째 적재가 만든 ID 목록으로만 한다.

In [1]:
from neo4j.exceptions import ConstraintError
from rag.prevention_legal.retrieve import load_test_blocks, delete_test_blocks
DOCS = {"kdca_measles_2026"}
first = []
try:
    first = load_test_blocks(DOCS)
    print("첫 적재:", len(first))
    try:
        second = load_test_blocks(DOCS)
        print("두 번째 적재가 성공함 (문제):", len(second))
    except ConstraintError as e:
        # 기대한 실패만 통과시킨다. 다른 오류는 그대로 올려 보낸다.
        print("두 번째 적재 실패 (기대):", type(e).__name__)
finally:
    print("삭제 개수:", delete_test_blocks(first))


첫 적재: 12


두 번째 적재 실패 (기대): ConstraintError


삭제 개수: 12


In [2]:
from rag.prevention_legal.neo4j_conn import connect
d = connect()
with d.session() as s:
    print("PLTest:", s.run("MATCH (c:PLTest) RETURN count(c) AS n").single()["n"])
    print("전체 Chunk:", s.run("MATCH (c:Chunk) RETURN count(c) AS n").single()["n"])
d.close()


PLTest: 0
전체 Chunk: 1569
